# Coherent bus and car interiors: fair half-cut comparison

This notebook shows the **15 buses and 15 cars with the largest precision-weighted semantic improvement** from Objective 1 to coherent single-donor retrieval. Retrieval votes select complete donor surface components; they never generate voxelwise consensus geometry.

Ground Truth, Objective 1 using view 18, and Coherent Retrieval are all rendered from the same 64³ voxel representation, passed through the same display-only smoothing, cut through the same plane, and shown with one synchronized camera. Change `SAMPLE_INDEX` to inspect all 30 examples.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pyvista as pv
from IPython.display import display
from PIL import Image

RESULTS_DIR = Path("results/coherent_retrieval_view18")
OBJECTIVE1_DIR = Path("results/objective1_view18_full/predictions/objective1/seed_42/voxels")
GROUND_TRUTH_DIR = RESULTS_DIR / "references" / "ground_truth"
FINAL_DIR = RESULTS_DIR / "predictions" / "coherent_retrieval"
RESOLUTION = 64
METRIC_MARGIN = 2
SAMPLES_PER_CATEGORY = 15
CATEGORIES = ("bus", "car")

# The positive side of this axis is removed. Try y or z for another cut.
CUT_NORMAL = np.array([1.0, 0.0, 0.0])
CUT_FRACTION = 0.5

METHODS = [
    ("Ground truth", "#d9dde5"),
    ("Objective 1 (view 18)", "#f4a261"),
    ("Coherent retrieval", "#55bde8"),
]

for required in (RESULTS_DIR / "per_sample.csv", RESULTS_DIR / "visualization_manifest.csv", OBJECTIVE1_DIR, GROUND_TRUTH_DIR, FINAL_DIR):
    if not required.exists():
        raise FileNotFoundError(f"Missing required result source: {required}")

metrics = {}
with (RESULTS_DIR / "per_sample.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        if (
            row["category"] in CATEGORIES
            and row["method"] in {"objective1", "coherent_retrieval"}
            and int(row["margin"]) == METRIC_MARGIN
        ):
            metrics.setdefault(row["sample_id"], {})[row["method"]] = {
                "category": row["category"],
                "internal_precision": float(row["internal_precision"]),
                "internal_recall": float(row["internal_recall"]),
                "internal_f1": float(row["internal_f1"]),
                "exterior_iou": float(row["exterior_iou"]),
                "internal_f05": float(row["internal_f05"]),
                "internal_ratio": float(row["pred_to_gt_internal_ratio"]),
                "components": int(row["internal_components_26"]),
                "solid_core_fraction": float(row["volumetric_core_fraction"]),
                "selection_k": int(row["selection_k"]),
                "support_k": int(row["support_k"]),
                "preset": row["preset"],
            }

for sample_id in list(metrics):
    if set(metrics[sample_id]) != {"objective1", "coherent_retrieval"}:
        del metrics[sample_id]

def f1_delta(sample_id):
    return (
        metrics[sample_id]["coherent_retrieval"]["internal_f1"]
        - metrics[sample_id]["objective1"]["internal_f1"]
    )

sample_ids = []
with (RESULTS_DIR / "visualization_manifest.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        if row["category"] in CATEGORIES:
            sample_ids.append(row["sample_id"])
expected_count = SAMPLES_PER_CATEGORY * len(CATEGORIES)
if len(sample_ids) != expected_count:
    raise ValueError(f"Expected {expected_count} gallery samples, found {len(sample_ids)}")

for sample_id in sample_ids:
    for path in (
        GROUND_TRUTH_DIR / f"{sample_id}.ply",
        OBJECTIVE1_DIR / f"{sample_id}.ply",
        FINAL_DIR / f"{sample_id}.ply",
    ):
        if not path.is_file():
            raise FileNotFoundError(f"Missing mesh source: {path}")

print(f"Verified {len(sample_ids)} coherent bus/car comparisons ({SAMPLES_PER_CATEGORY} per category)")
for index, sample_id in enumerate(sample_ids):
    category = metrics[sample_id]["objective1"]["category"]
    print(f"  {index:02d}: {category:3s} delta {f1_delta(sample_id):+.3f} | {sample_id}")


In [ ]:
def read_voxels(path):
    points = np.asarray(pv.read(path).points, dtype=np.float32).reshape(-1, 3)
    if len(points) == 0:
        return np.empty((0, 3), dtype=np.int32)
    coordinates = np.floor((points + 0.5) * RESOLUTION).astype(np.int32)
    return np.unique(np.clip(coordinates, 0, RESOLUTION - 1), axis=0)


def voxel_surface(coordinates):
    occupancy = np.zeros((RESOLUTION, RESOLUTION, RESOLUTION), dtype=np.float32)
    occupancy[tuple(coordinates.T)] = 1.0
    field = np.pad(occupancy, 1)
    spacing = 1.0 / RESOLUTION
    grid = pv.ImageData(
        dimensions=field.shape,
        spacing=(spacing, spacing, spacing),
        origin=(-0.5 - 0.5 * spacing,) * 3,
    )
    grid.point_data["occupancy"] = field.ravel(order="F")
    surface = grid.contour([0.5], scalars="occupancy").clean().triangulate()
    # Display-only smoothing is identical for GT and both predictions.
    return surface.smooth(n_iter=30, relaxation_factor=0.04, boundary_smoothing=False)


def camera_for_cut(normal, center, object_size):
    up = np.array([0.0, 0.0, 1.0])
    if abs(normal @ up) > 0.9:
        up = np.array([0.0, 1.0, 0.0])
    side = np.cross(up, normal)
    position = center + object_size * (2.0 * normal + 0.65 * side + 0.45 * up)
    return [position.tolist(), center.tolist(), up.tolist()]


def render_comparison(sample_id):
    meshes = {
        "Ground truth": voxel_surface(read_voxels(GROUND_TRUTH_DIR / f"{sample_id}.ply")),
        "Objective 1 (view 18)": voxel_surface(read_voxels(OBJECTIVE1_DIR / f"{sample_id}.ply")),
        "Coherent retrieval": voxel_surface(read_voxels(FINAL_DIR / f"{sample_id}.ply")),
    }
    ground_truth = meshes["Ground truth"]
    normal = CUT_NORMAL / np.linalg.norm(CUT_NORMAL)
    projection = np.asarray(ground_truth.points) @ normal
    cut_offset = projection.max() - CUT_FRACTION * np.ptp(projection)
    cut_origin = normal * cut_offset
    center = np.asarray(ground_truth.center)
    object_size = ground_truth.length

    plotter = pv.Plotter(shape=(1, 3), off_screen=True, window_size=(1800, 650))
    try:
        plotter.enable_anti_aliasing("ssaa")
    except Exception:
        pass

    for column, (label, color) in enumerate(METHODS):
        cut_mesh = meshes[label].clip(normal=normal, origin=cut_origin, invert=True)
        plotter.subplot(0, column)
        plotter.set_background("white")
        plotter.add_mesh(
            cut_mesh,
            color=color,
            smooth_shading=True,
            ambient=0.28,
            diffuse=0.75,
            specular=0.12,
            show_edges=False,
        )
        plotter.add_text(label, position="upper_left", color="black", font_size=11)

    plotter.link_views()
    plotter.camera_position = camera_for_cut(normal, center, object_size)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = 0.58 * object_size
    image = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(image)


In [ ]:
SAMPLE_INDEX = 0  # 0-14 are buses; 15-29 are cars.

sample_id = sample_ids[SAMPLE_INDEX]
baseline = metrics[sample_id]["objective1"]
final = metrics[sample_id]["coherent_retrieval"]
print(f"{sample_id} | {baseline['category']}")
print(
    f"Frozen coherent policy: donor top-{final['selection_k']}, "
    f"support pool={final['support_k']}, preset={final['preset']}"
)
print(
    f"Margin-{METRIC_MARGIN} internal F1: "
    f"{baseline['internal_f1']:.3f} -> {final['internal_f1']:.3f} "
    f"(delta {f1_delta(sample_id):+.3f})"
)
print(
    f"Precision: {baseline['internal_precision']:.3f} -> {final['internal_precision']:.3f} | "
    f"Recall: {baseline['internal_recall']:.3f} -> {final['internal_recall']:.3f}"
)
print(
    f"Internal ratio: {baseline['internal_ratio']:.2f} -> {final['internal_ratio']:.2f} | "
    f"26-connected components: {baseline['components']} -> {final['components']}"
)
print(
    f"Volumetric-core fraction: {baseline['solid_core_fraction']:.3f} -> "
    f"{final['solid_core_fraction']:.3f} (lower is more surface-like)"
)
display(render_comparison(sample_id))


In [ ]:
SHOW_ALL = True

if SHOW_ALL:
    for index, sample_id in enumerate(sample_ids):
        baseline = metrics[sample_id]["objective1"]
        final = metrics[sample_id]["coherent_retrieval"]
        print(
            f"{index:02d}. {sample_id} | F1 "
            f"{baseline['internal_f1']:.3f} -> {final['internal_f1']:.3f} "
            f"(delta {f1_delta(sample_id):+.3f})"
        )
        display(render_comparison(sample_id))
